# total parametres in each

In [15]:

from pathlib import Path
import subprocess
import sys

base = Path("/home/cse-sdpl/Downloads")

experiments = {
    "E0": base / "CMX_Original",
    "E1": base / "CMX_E1",
    "E2": base / "CMX_E2",
    "E3": base / "CMX_E3",
    "E4": base / "CMX_E4",
    "E5": base / "CMX_E5",
}

count_code = r'''
from config import config as C
from models.builder import EncoderDecoder

model = EncoderDecoder(cfg=C)

total = sum(p.numel() for p in model.parameters())
trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Backbone: {C.backbone}")
print(f"Total: {total}")
print(f"Trainable: {trainable}")
'''

results = []

for exp, project in experiments.items():
    print(f"\n{'=' * 45}")
    print(f"Experiment: {exp}")
    print(f"Project: {project}")

    if not project.is_dir():
        print("STATUS: Project directory not found")
        continue

    try:
        run = subprocess.run(
            [sys.executable, "-c", count_code],
            cwd=str(project),
            capture_output=True,
            text=True,
            timeout=180,
        )

        if run.returncode != 0:
            print("STATUS: ERROR")
            print(run.stderr[-2500:])
            continue

        print(run.stdout.strip())

        values = {}
        for line in run.stdout.splitlines():
            if ":" in line:
                key, value = line.split(":", 1)
                values[key.strip()] = value.strip()

        total = int(values["Total"])
        trainable = int(values["Trainable"])

        results.append({
            "Experiment": exp,
            "Backbone": values["Backbone"],
            "Total parameters": f"{total:,}",
            "Total (M)": f"{total / 1e6:.3f}",
            "Trainable parameters": f"{trainable:,}",
            "Status": "OK",
        })

    except Exception as e:
        print(f"STATUS: ERROR - {e}")

import pandas as pd

results_df = pd.DataFrame(results)
display(results_df)



Experiment: E0
Project: /home/cse-sdpl/Downloads/CMX_Original
Backbone: mit_b2
Total: 66565521
Trainable: 66565521

Experiment: E1
Project: /home/cse-sdpl/Downloads/CMX_E1
Backbone: mit_b2_e1
Total: 66565521
Trainable: 66565521

Experiment: E2
Project: /home/cse-sdpl/Downloads/CMX_E2
Backbone: mit_b2_e2
Total: 66565529
Trainable: 66565529

Experiment: E3
Project: /home/cse-sdpl/Downloads/CMX_E3
Backbone: mit_b2_e3
Total: 66662169
Trainable: 66662169

Experiment: E4
Project: /home/cse-sdpl/Downloads/CMX_E4
Backbone: mit_b2_e4
Total: 66758817
Trainable: 66758817

Experiment: E5
Project: /home/cse-sdpl/Downloads/CMX_E5
Backbone: mit_b2_e5
Total: 66758817
Trainable: 66758817


,Experiment,Backbone,Total parameters,Total (M),Trainable parameters,Status
0,E0,mit_b2,"66,565,521",66.566,"66,565,521",OK
1,E1,mit_b2_e1,"66,565,521",66.566,"66,565,521",OK
2,E2,mit_b2_e2,"66,565,529",66.566,"66,565,529",OK
3,E3,mit_b2_e3,"66,662,169",66.662,"66,662,169",OK
4,E4,mit_b2_e4,"66,758,817",66.759,"66,758,817",OK
5,E5,mit_b2_e5,"66,758,817",66.759,"66,758,817",OK


In [1]:

from pathlib import Path

# Update this if your experiments are stored elsewhere
BASE = Path("/home/cse-sdpl/Downloads")

experiments = ["CMX_Original", "CMX_E1", "CMX_E2", "CMX_E3", "CMX_E4"]

for exp in experiments:
    root = BASE / exp
    print(f"\n{'='*50}\n{exp}")
    print("Folder exists:", root.exists())

    if root.exists():
        checkpoints = list(root.rglob("epoch-100.pth"))
        for ckpt in checkpoints:
            print(f"Checkpoint: {ckpt}")
            print(f"Size: {ckpt.stat().st_size / (1024**3):.2f} GB")

        if not checkpoints:
            print("epoch-100.pth not found; checking other checkpoint files...")
            for ckpt in list(root.rglob("*.pth"))[:10]:
                print(ckpt)



CMX_Original
Folder exists: True
Checkpoint: /home/cse-sdpl/Downloads/CMX_Original/log_TDD_mit_b2/checkpoint/epoch-100.pth
Size: 0.74 GB

CMX_E1
Folder exists: True
Checkpoint: /home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2_e1/checkpoint/epoch-100.pth
Size: 0.67 GB

CMX_E2
Folder exists: True
Checkpoint: /home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e1/checkpoint/epoch-100.pth
Size: 0.67 GB
Checkpoint: /home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth
Size: 0.74 GB

CMX_E3
Folder exists: True
Checkpoint: /home/cse-sdpl/Downloads/CMX_E3/log_TDD_mit_b2_e1/checkpoint/epoch-100.pth
Size: 0.67 GB
Checkpoint: /home/cse-sdpl/Downloads/CMX_E3/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth
Size: 0.74 GB
Checkpoint: /home/cse-sdpl/Downloads/CMX_E3/log_TDD_mit_b2_e3/checkpoint/epoch-100.pth
Size: 0.75 GB

CMX_E4
Folder exists: True
Checkpoint: /home/cse-sdpl/Downloads/CMX_E4/log_TDD_mit_b2_e4/checkpoint/epoch-100.pth
Size: 0.75 GB


In [2]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_Original")

!python eval.py -e 100 -d 0 --split test -p predictions_E0_test


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:32:59 using devices 0
16 02:32:59 Using backbone: Segformer-B2
16 02:33:00 Using MLP Decoder
16 02:33:00 Load Model: /home/cse-sdpl/Downloads/CMX_Original/log_TDD_mit_b2/checkpoint/epoch-100.pth
16 02:33:04 Load model, Time usage:
	IO: 4.423196315765381, initialize parameters: 0.04419755935668945
16 02:33:04 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@8.862] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:33:06 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:11<01:02,  5.36it/s]16 02:33:16 Save the image 01459D.png


 31%|████████████▍                           | 122/393 [00:23<00:50,  5.37it/s]16 02:33:28 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:35<00:37,  5.45it/s]16 02:33:40 Save the image 01589D.png


 64%|█████████████████████████▋              | 252/393 [00:47<00:25,  5.48it/s]16 02:33:52 Save the image 01236N.png


 81%|████████████████████████████████▎       | 317/393 [00:59<00:13,  5.45it/s]16 02:34:03 Save the image 01301N.png


 97%|██████████████████████████████████████▉ | 382/393 [01:10<00:01,  5.60it/s]16 02:34:15 Save the image 01366N.png


100%|████████████████████████████████████████| 393/393 [01:12<00:00,  5.45it/s]
1 unlabeled	98.303%
2 car   	89.098%
3 person	75.033%
4 bike  	64.802%
5 curve 	47.548%
6 car_stop	33.639%
7 guardrail	9.273%
8 color_cone	51.472%
9 bump  	61.139%
----------     mean_IoU	58.923%	freq_IoU	96.777%	mean_pixel_acc	70.841%	pixel_acc	98.227%
16 02:34:16 Evaluation Elapsed Time: 72.15s


In [3]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_Original")

!python eval.py -e 100 -d 0 --split val -p predictions_E0_val


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:34:58 using devices 0
16 02:34:58 Using backbone: Segformer-B2
16 02:34:59 Using MLP Decoder
16 02:35:00 Load Model: /home/cse-sdpl/Downloads/CMX_Original/log_TDD_mit_b2/checkpoint/epoch-100.pth
16 02:35:04 Load model, Time usage:
	IO: 3.993239402770996, initialize parameters: 0.043116092681884766
16 02:35:04 GPU 0 handle 392 data.
  0%|                                                  | 0/392 [00:00<?, ?it/s][ WARN:0@9.322] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:35:05 Save the image 00498D.png
 15%|█████▉                                   | 57/392 [00:11<00:59,  5.65it/s]16 02:35:16 Save the image 00555D.png


 31%|████████████▍                           | 122/392 [00:23<00:50,  5.38it/s]16 02:35:27 Save the image 00620D.png


 48%|███████████████████                     | 187/392 [00:35<00:37,  5.46it/s]16 02:35:39 Save the image 01384D.png


 64%|█████████████████████████▋              | 252/392 [00:47<00:25,  5.57it/s]16 02:35:51 Save the image 01049N.png


 81%|████████████████████████████████▎       | 317/392 [00:58<00:13,  5.57it/s]16 02:36:03 Save the image 01114N.png


 97%|██████████████████████████████████████▉ | 382/392 [01:10<00:01,  5.59it/s]16 02:36:15 Save the image 01179N.png


100%|████████████████████████████████████████| 392/392 [01:12<00:00,  5.41it/s]
1 unlabeled	97.775%
2 car   	87.045%
3 person	71.804%
4 bike  	72.866%
5 curve 	38.150%
6 car_stop	39.068%
7 guardrail	48.623%
8 color_cone	58.333%
9 bump  	50.866%
----------     mean_IoU	62.726%	freq_IoU	95.784%	mean_pixel_acc	72.982%	pixel_acc	97.732%
16 02:36:16 Evaluation Elapsed Time: 72.43s


In [4]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_Original")

!python eval.py -e 100 -d 0 --split train -p predictions_E0_train


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:36:48 using devices 0
16 02:36:48 Using backbone: Segformer-B2
16 02:36:49 Using MLP Decoder
16 02:36:49 Load Model: /home/cse-sdpl/Downloads/CMX_Original/log_TDD_mit_b2/checkpoint/epoch-100.pth
16 02:36:53 Load model, Time usage:
	IO: 4.055520057678223, initialize parameters: 0.04480147361755371
16 02:36:53 GPU 0 handle 1568 data.
  0%|                                                 | 0/1568 [00:00<?, ?it/s][ WARN:0@8.409] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:36:55 Save the image 00001D.png
  4%|█▍                                      | 56/1568 [00:11<04:28,  5.63it/s]16 02:37:05 Save the image 00105D.png


  8%|██▉                                    | 120/1568 [00:22<04:10,  5.78it/s]16 02:37:16 Save the image 00137D.png


 12%|████▌                                  | 184/1568 [00:33<03:57,  5.83it/s]16 02:37:27 Save the image 00169D.png


 16%|██████▏                                | 248/1568 [00:44<03:49,  5.75it/s]16 02:37:38 Save the image 00201D.png


 20%|███████▊                               | 312/1568 [00:55<03:38,  5.76it/s]16 02:37:49 Save the image 00233D.png


...

 85%|████████████████████████████████▍     | 1336/1568 [03:50<00:38,  5.98it/s]16 02:40:44 Save the image 00886N.png


 89%|█████████████████████████████████▉    | 1400/1568 [04:01<00:28,  5.88it/s]16 02:40:55 Save the image 00918N.png


 93%|███████████████████████████████████▍  | 1464/1568 [04:12<00:17,  5.78it/s]16 02:41:06 Save the image 00950N.png


 97%|█████████████████████████████████████ | 1528/1568 [04:23<00:06,  5.76it/s]16 02:41:17 Save the image 00982N.png


100%|██████████████████████████████████████| 1568/1568 [04:30<00:00,  5.79it/s]
1 unlabeled	99.520%
2 car   	97.232%
3 person	89.233%
4 bike  	90.006%
5 curve 	90.455%
6 car_stop	92.457%
7 guardrail	95.618%
8 color_cone	85.688%
9 bump  	94.267%
----------     mean_IoU	92.720%	freq_IoU	99.092%	mean_pixel_acc	96.772%	pixel_acc	99.535%
16 02:41:24 Evaluation Elapsed Time: 270.66s


# e1

In [9]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E1")

!python eval.py -e 100 -d 0 --split test -p predictions_E1_test


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:44:41 using devices 0
16 02:44:41 Using E1: SegFormer-B2 without FRM
16 02:44:42 Using MLP Decoder
16 02:44:42 Load Model: /home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2_e1/checkpoint/epoch-100.pth
16 02:44:46 Load model, Time usage:
	IO: 3.9483625888824463, initialize parameters: 0.043245553970336914
16 02:44:46 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@8.194] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:44:47 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:10<00:57,  5.80it/s]16 02:44:57 Save the image 01459D.png


 31%|████████████▍                           | 122/393 [00:22<00:46,  5.83it/s]16 02:45:08 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:32<00:33,  6.17it/s]16 02:45:19 Save the image 01589D.png


 64%|█████████████████████████▋              | 252/393 [00:43<00:23,  5.99it/s]16 02:45:30 Save the image 01236N.png


 81%|████████████████████████████████▎       | 317/393 [00:54<00:12,  6.07it/s]16 02:45:40 Save the image 01301N.png


 97%|██████████████████████████████████████▉ | 382/393 [01:05<00:01,  5.81it/s]16 02:45:51 Save the image 01366N.png


100%|████████████████████████████████████████| 393/393 [01:07<00:00,  5.82it/s]
1 unlabeled	98.237%
2 car   	88.676%
3 person	74.412%
4 bike  	64.871%
5 curve 	46.047%
6 car_stop	29.591%
7 guardrail	6.372%
8 color_cone	44.308%
9 bump  	58.564%
----------     mean_IoU	56.786%	freq_IoU	96.664%	mean_pixel_acc	65.979%	pixel_acc	98.176%
16 02:45:53 Evaluation Elapsed Time: 67.54s


In [10]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E1")

!python eval.py -e 100 -d 0 --split val -p predictions_E1_val


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:46:17 using devices 0
16 02:46:17 Using E1: SegFormer-B2 without FRM
16 02:46:18 Using MLP Decoder
16 02:46:20 Load Model: /home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2_e1/checkpoint/epoch-100.pth
16 02:46:24 Load model, Time usage:
	IO: 4.102044343948364, initialize parameters: 0.042738914489746094
16 02:46:24 GPU 0 handle 392 data.
  0%|                                                  | 0/392 [00:00<?, ?it/s][ WARN:0@10.188] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:46:25 Save the image 00498D.png
 15%|█████▉                                   | 57/392 [00:10<00:56,  5.98it/s]16 02:46:35 Save the image 00555D.png


 31%|████████████▍                           | 122/392 [00:21<00:45,  5.96it/s]16 02:46:46 Save the image 00620D.png


 48%|███████████████████                     | 187/392 [00:32<00:34,  5.94it/s]16 02:46:57 Save the image 01384D.png


 64%|█████████████████████████▋              | 252/392 [00:43<00:23,  5.88it/s]16 02:47:08 Save the image 01049N.png


 81%|████████████████████████████████▎       | 317/392 [00:54<00:12,  5.94it/s]16 02:47:19 Save the image 01114N.png


 97%|██████████████████████████████████████▉ | 382/392 [01:05<00:01,  5.79it/s]16 02:47:30 Save the image 01179N.png


100%|████████████████████████████████████████| 392/392 [01:07<00:00,  5.82it/s]
1 unlabeled	97.726%
2 car   	87.001%
3 person	71.608%
4 bike  	72.682%
5 curve 	36.603%
6 car_stop	32.581%
7 guardrail	45.387%
8 color_cone	56.540%
9 bump  	48.367%
----------     mean_IoU	60.944%	freq_IoU	95.678%	mean_pixel_acc	70.880%	pixel_acc	97.684%
16 02:47:31 Evaluation Elapsed Time: 67.38s


In [11]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E1")

!python eval.py -e 100 -d 0 --split train -p predictions_E1_train


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:49:04 using devices 0
16 02:49:04 Using E1: SegFormer-B2 without FRM
16 02:49:05 Using MLP Decoder
16 02:49:05 Load Model: /home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2_e1/checkpoint/epoch-100.pth
16 02:49:09 Load model, Time usage:
	IO: 4.073198080062866, initialize parameters: 0.04323744773864746
16 02:49:09 GPU 0 handle 1568 data.
  0%|                                                 | 0/1568 [00:00<?, ?it/s][ WARN:0@8.131] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:49:11 Save the image 00001D.png
  4%|█▍                                      | 56/1568 [00:10<04:21,  5.79it/s]16 02:49:20 Save the image 00105D.png


  8%|██▉                                    | 120/1568 [00:21<03:57,  6.09it/s]16 02:49:31 Save the image 00137D.png


 12%|████▌                                  | 184/1568 [00:32<03:47,  6.10it/s]16 02:49:42 Save the image 00169D.png


 16%|██████▏                                | 248/1568 [00:42<03:39,  6.00it/s]16 02:49:52 Save the image 00201D.png


 20%|███████▊                               | 312/1568 [00:53<03:26,  6.08it/s]16 02:50:03 Save the image 00233D.png


...

 85%|████████████████████████████████▍     | 1336/1568 [03:46<00:38,  5.95it/s]16 02:52:55 Save the image 00886N.png


 89%|█████████████████████████████████▉    | 1400/1568 [03:56<00:28,  5.99it/s]16 02:53:06 Save the image 00918N.png


 93%|███████████████████████████████████▍  | 1464/1568 [04:07<00:17,  5.82it/s]16 02:53:17 Save the image 00950N.png


 97%|█████████████████████████████████████ | 1528/1568 [04:18<00:06,  5.91it/s]16 02:53:28 Save the image 00982N.png


100%|██████████████████████████████████████| 1568/1568 [04:25<00:00,  5.91it/s]
1 unlabeled	99.543%
2 car   	97.360%
3 person	89.765%
4 bike  	90.399%
5 curve 	90.996%
6 car_stop	92.626%
7 guardrail	95.882%
8 color_cone	86.249%
9 bump  	94.563%
----------     mean_IoU	93.043%	freq_IoU	99.134%	mean_pixel_acc	96.913%	pixel_acc	99.557%
16 02:53:35 Evaluation Elapsed Time: 265.40s


# e2

In [12]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E2")

!python eval.py -e 100 -d 0 --split test -p predictions_E2_test


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:54:58 using devices 0
16 02:54:58 Using E2: SegFormer-B2 with FRM and weighted cross-attention fusion
16 02:54:59 Using MLP Decoder
16 02:54:59 Load Model: /home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth
16 02:55:04 Load model, Time usage:
	IO: 4.514350414276123, initialize parameters: 0.04472064971923828
16 02:55:04 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@8.795] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:55:05 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:11<00:57,  5.84it/s]16 02:55:15

 31%|████████████▍                           | 122/393 [00:22<00:47,  5.73it/s]16 02:55:26 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:33<00:35,  5.82it/s]16 02:55:38 Save the image 01589D.png


 64%|█████████████████████████▋              | 252/393 [00:44<00:23,  6.08it/s]16 02:55:49 Save the image 01236N.png


 81%|████████████████████████████████▎       | 317/393 [00:55<00:12,  5.94it/s]16 02:56:00 Save the image 01301N.png


 97%|██████████████████████████████████████▉ | 382/393 [01:07<00:01,  5.89it/s]16 02:56:11 Save the image 01366N.png


100%|████████████████████████████████████████| 393/393 [01:09<00:00,  5.69it/s]
1 unlabeled	98.296%
2 car   	89.336%
3 person	75.209%
4 bike  	64.498%
5 curve 	49.209%
6 car_stop	29.546%
7 guardrail	6.311%
8 color_cone	52.058%
9 bump  	56.012%
----------     mean_IoU	57.830%	freq_IoU	96.768%	mean_pixel_acc	67.892%	pixel_acc	98.224%
16 02:56:13 Evaluation Elapsed Time: 69.06s


In [13]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E2")

!python eval.py -e 100 -d 0 --split val -p predictions_E2_val


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:57:23 using devices 0
16 02:57:23 Using E2: SegFormer-B2 with FRM and weighted cross-attention fusion
16 02:57:24 Using MLP Decoder
16 02:57:24 Load Model: /home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth
16 02:57:28 Load model, Time usage:
	IO: 4.229828357696533, initialize parameters: 0.04425406455993652
16 02:57:28 GPU 0 handle 392 data.
  0%|                                                  | 0/392 [00:00<?, ?it/s][ WARN:0@10.499] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:57:32 Save the image 00498D.png
 15%|█████▉                                   | 57/392 [00:13<00:58,  5.72it/s]16 02:57:4

 31%|████████████▍                           | 122/392 [00:24<00:47,  5.72it/s]16 02:57:53 Save the image 00620D.png


 48%|███████████████████                     | 187/392 [00:35<00:36,  5.68it/s]16 02:58:04 Save the image 01384D.png


 64%|█████████████████████████▋              | 252/392 [00:46<00:24,  5.76it/s]16 02:58:15 Save the image 01049N.png


 81%|████████████████████████████████▎       | 317/392 [00:58<00:12,  5.92it/s]16 02:58:27 Save the image 01114N.png


 97%|██████████████████████████████████████▉ | 382/392 [01:09<00:01,  5.77it/s]16 02:58:38 Save the image 01179N.png


100%|████████████████████████████████████████| 392/392 [01:11<00:00,  5.51it/s]
1 unlabeled	97.783%
2 car   	86.555%
3 person	71.449%
4 bike  	73.042%
5 curve 	38.948%
6 car_stop	41.058%
7 guardrail	45.470%
8 color_cone	58.579%
9 bump  	48.038%
----------     mean_IoU	62.325%	freq_IoU	95.773%	mean_pixel_acc	72.627%	pixel_acc	97.731%
16 02:58:39 Evaluation Elapsed Time: 71.16s


In [14]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E2")

!python eval.py -e 100 -d 0 --split train -p predictions_E2_train


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 02:59:31 using devices 0
16 02:59:31 Using E2: SegFormer-B2 with FRM and weighted cross-attention fusion
16 02:59:32 Using MLP Decoder
16 02:59:32 Load Model: /home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2_e2/checkpoint/epoch-100.pth
16 02:59:36 Load model, Time usage:
	IO: 4.218412637710571, initialize parameters: 0.0432584285736084
16 02:59:36 GPU 0 handle 1568 data.
  0%|                                                 | 0/1568 [00:00<?, ?it/s][ WARN:0@8.359] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 02:59:38 Save the image 00001D.png
  4%|█▍                                      | 56/1568 [00:10<04:17,  5.86it/s]16 02:59:47

  8%|██▉                                    | 120/1568 [00:21<04:10,  5.78it/s]16 02:59:58 Save the image 00137D.png


 12%|████▌                                  | 184/1568 [00:32<03:56,  5.85it/s]16 03:00:09 Save the image 00169D.png


 16%|██████▏                                | 248/1568 [00:43<03:47,  5.81it/s]16 03:00:20 Save the image 00201D.png


 20%|███████▊                               | 312/1568 [00:54<03:29,  5.99it/s]16 03:00:31 Save the image 00233D.png


...

 85%|████████████████████████████████▍     | 1336/1568 [03:50<00:40,  5.70it/s]16 03:03:27 Save the image 00886N.png


 89%|█████████████████████████████████▉    | 1400/1568 [04:00<00:26,  6.30it/s]16 03:03:37 Save the image 00918N.png


 93%|███████████████████████████████████▍  | 1464/1568 [04:11<00:18,  5.72it/s]16 03:03:48 Save the image 00950N.png


 97%|█████████████████████████████████████ | 1528/1568 [04:22<00:06,  5.83it/s]16 03:03:59 Save the image 00982N.png


100%|██████████████████████████████████████| 1568/1568 [04:29<00:00,  5.82it/s]
1 unlabeled	99.514%
2 car   	97.196%
3 person	89.110%
4 bike  	89.782%
5 curve 	90.328%
6 car_stop	92.313%
7 guardrail	95.834%
8 color_cone	85.626%
9 bump  	94.115%
----------     mean_IoU	92.646%	freq_IoU	99.080%	mean_pixel_acc	96.752%	pixel_acc	99.528%
16 03:04:06 Evaluation Elapsed Time: 269.48s


# E3

In [16]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E3")

!python eval.py -e 100 -d 0 --split test -p predictions_E3_test


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 03:05:26 using devices 0
16 03:05:26 Using E3: SegFormer-B2 with reliability-guided fusion
16 03:05:27 Using MLP Decoder
16 03:05:27 Load Model: /home/cse-sdpl/Downloads/CMX_E3/log_TDD_mit_b2_e3/checkpoint/epoch-100.pth
16 03:05:32 Load model, Time usage:
	IO: 4.592591047286987, initialize parameters: 0.0491793155670166
16 03:05:32 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@8.862] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 03:05:33 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:11<01:02,  5.35it/s]16 03:05:44 Save the image

 31%|████████████▍                           | 122/393 [00:23<00:49,  5.52it/s]16 03:05:56 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:35<00:38,  5.38it/s]16 03:06:08 Save the image 01589D.png


 64%|█████████████████████████▋              | 252/393 [00:47<00:25,  5.50it/s]16 03:06:20 Save the image 01236N.png


 81%|████████████████████████████████▎       | 317/393 [00:59<00:13,  5.49it/s]16 03:06:32 Save the image 01301N.png


 97%|██████████████████████████████████████▉ | 382/393 [01:11<00:02,  5.43it/s]16 03:06:44 Save the image 01366N.png


100%|████████████████████████████████████████| 393/393 [01:13<00:00,  5.35it/s]
1 unlabeled	98.314%
2 car   	89.556%
3 person	75.183%
4 bike  	64.706%
5 curve 	47.586%
6 car_stop	30.836%
7 guardrail	6.209%
8 color_cone	51.353%
9 bump  	57.304%
----------     mean_IoU	57.894%	freq_IoU	96.792%	mean_pixel_acc	67.030%	pixel_acc	98.247%
16 03:06:46 Evaluation Elapsed Time: 73.52s


In [17]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E3")

!python eval.py -e 100 -d 0 --split val -p predictions_E3_val


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 03:07:12 using devices 0
16 03:07:12 Using E3: SegFormer-B2 with reliability-guided fusion
16 03:07:13 Using MLP Decoder
16 03:07:15 Load Model: /home/cse-sdpl/Downloads/CMX_E3/log_TDD_mit_b2_e3/checkpoint/epoch-100.pth
16 03:07:19 Load model, Time usage:
	IO: 4.246111869812012, initialize parameters: 0.04618048667907715
16 03:07:19 GPU 0 handle 392 data.
  0%|                                                  | 0/392 [00:00<?, ?it/s][ WARN:0@10.523] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 03:07:21 Save the image 00498D.png
 15%|█████▉                                   | 57/392 [00:12<01:03,  5.29it/s]16 03:07:32 Save the ima

 31%|████████████▍                           | 122/392 [00:23<00:49,  5.42it/s]16 03:07:44 Save the image 00620D.png


 48%|███████████████████                     | 187/392 [00:35<00:38,  5.39it/s]16 03:07:56 Save the image 01384D.png


 64%|█████████████████████████▋              | 252/392 [00:47<00:25,  5.40it/s]16 03:08:07 Save the image 01049N.png


 81%|████████████████████████████████▎       | 317/392 [00:59<00:13,  5.45it/s]16 03:08:19 Save the image 01114N.png


 97%|██████████████████████████████████████▉ | 382/392 [01:11<00:01,  5.57it/s]16 03:08:31 Save the image 01179N.png


100%|████████████████████████████████████████| 392/392 [01:13<00:00,  5.34it/s]
1 unlabeled	97.769%
2 car   	86.617%
3 person	71.802%
4 bike  	72.749%
5 curve 	38.595%
6 car_stop	35.661%
7 guardrail	48.186%
8 color_cone	56.572%
9 bump  	49.702%
----------     mean_IoU	61.961%	freq_IoU	95.739%	mean_pixel_acc	71.182%	pixel_acc	97.723%
16 03:08:33 Evaluation Elapsed Time: 73.42s


In [18]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E3")

!python eval.py -e 100 -d 0 --split train -p predictions_E3_train


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 03:11:14 using devices 0
16 03:11:14 Using E3: SegFormer-B2 with reliability-guided fusion
16 03:11:15 Using MLP Decoder
16 03:11:15 Load Model: /home/cse-sdpl/Downloads/CMX_E3/log_TDD_mit_b2_e3/checkpoint/epoch-100.pth
16 03:11:19 Load model, Time usage:
	IO: 4.348769187927246, initialize parameters: 0.044000864028930664
16 03:11:19 GPU 0 handle 1568 data.
  0%|                                                 | 0/1568 [00:00<?, ?it/s][ WARN:0@8.676] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 03:11:21 Save the image 00001D.png
  4%|█▍                                      | 56/1568 [00:11<04:38,  5.44it/s]16 03:11:31 Save the im

  8%|██▉                                    | 120/1568 [00:23<04:17,  5.63it/s]16 03:11:43 Save the image 00137D.png


 12%|████▌                                  | 184/1568 [00:35<04:11,  5.50it/s]16 03:11:54 Save the image 00169D.png


 16%|██████▏                                | 248/1568 [00:46<04:01,  5.47it/s]16 03:12:06 Save the image 00201D.png


 20%|███████▊                               | 312/1568 [00:58<03:48,  5.49it/s]16 03:12:18 Save the image 00233D.png


...

 85%|████████████████████████████████▍     | 1336/1568 [04:05<00:43,  5.33it/s]16 03:15:25 Save the image 00886N.png


 89%|█████████████████████████████████▉    | 1400/1568 [04:17<00:30,  5.50it/s]16 03:15:37 Save the image 00918N.png


 93%|███████████████████████████████████▍  | 1464/1568 [04:29<00:19,  5.43it/s]16 03:15:49 Save the image 00950N.png


 97%|█████████████████████████████████████ | 1528/1568 [04:41<00:07,  5.35it/s]16 03:16:01 Save the image 00982N.png


100%|██████████████████████████████████████| 1568/1568 [04:48<00:00,  5.43it/s]
1 unlabeled	99.659%
2 car   	97.970%
3 person	91.854%
4 bike  	92.752%
5 curve 	93.865%
6 car_stop	94.882%
7 guardrail	97.351%
8 color_cone	89.706%
9 bump  	96.198%
----------     mean_IoU	94.915%	freq_IoU	99.351%	mean_pixel_acc	97.607%	pixel_acc	99.670%
16 03:16:08 Evaluation Elapsed Time: 288.80s


# e4

In [20]:
import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E4")

!python eval.py -e 100 -d 0 --split test -p predictions_E4_test


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 03:17:58 using devices 0
16 03:17:58 Using E4: SegFormer-B2 with reliability-gated rectification
16 03:17:58 Using MLP Decoder
16 03:18:01 Load Model: /home/cse-sdpl/Downloads/CMX_E4/log_TDD_mit_b2_e4/checkpoint/epoch-100.pth
16 03:18:06 Load model, Time usage:
	IO: 4.23034930229187, initialize parameters: 0.04506731033325195
16 03:18:06 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@12.651] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 03:18:08 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:12<00:57,  5.89it/s]16 03:18:18 Save th

 31%|████████████▍                           | 122/393 [00:23<00:47,  5.74it/s]16 03:18:29 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:34<00:35,  5.77it/s]16 03:18:41 Save the image 01589D.png


 64%|█████████████████████████▋              | 252/393 [00:45<00:24,  5.76it/s]16 03:18:52 Save the image 01236N.png


 81%|████████████████████████████████▎       | 317/393 [00:57<00:13,  5.69it/s]16 03:19:03 Save the image 01301N.png


 97%|██████████████████████████████████████▉ | 382/393 [01:08<00:01,  5.82it/s]16 03:19:14 Save the image 01366N.png


100%|████████████████████████████████████████| 393/393 [01:10<00:00,  5.59it/s]
1 unlabeled	98.335%
2 car   	89.669%
3 person	75.300%
4 bike  	64.247%
5 curve 	46.909%
6 car_stop	32.645%
7 guardrail	4.883%
8 color_cone	52.549%
9 bump  	58.233%
----------     mean_IoU	58.086%	freq_IoU	96.816%	mean_pixel_acc	66.617%	pixel_acc	98.258%
16 03:19:16 Evaluation Elapsed Time: 70.32s


In [21]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E4")

!python eval.py -e 100 -d 0 --split val -p predictions_E4_val

/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 03:19:45 using devices 0
16 03:19:45 Using E4: SegFormer-B2 with reliability-gated rectification
16 03:19:46 Using MLP Decoder
16 03:19:46 Load Model: /home/cse-sdpl/Downloads/CMX_E4/log_TDD_mit_b2_e4/checkpoint/epoch-100.pth
16 03:19:51 Load model, Time usage:
	IO: 4.325042486190796, initialize parameters: 0.044379472732543945
16 03:19:51 GPU 0 handle 392 data.
  0%|                                                  | 0/392 [00:00<?, ?it/s][ WARN:0@8.458] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 03:19:52 Save the image 00498D.png
 15%|█████▉                                   | 57/392 [00:11<00:58,  5.75it/s]16 03:20:02 Save t

 31%|████████████▍                           | 122/392 [00:22<00:46,  5.84it/s]16 03:20:13 Save the image 00620D.png


 48%|███████████████████                     | 187/392 [00:33<00:36,  5.64it/s]16 03:20:24 Save the image 01384D.png


 64%|█████████████████████████▋              | 252/392 [00:44<00:24,  5.72it/s]16 03:20:36 Save the image 01049N.png


 81%|████████████████████████████████▎       | 317/392 [00:56<00:12,  5.77it/s]16 03:20:47 Save the image 01114N.png


 97%|██████████████████████████████████████▉ | 382/392 [01:07<00:01,  5.66it/s]16 03:20:58 Save the image 01179N.png


100%|████████████████████████████████████████| 392/392 [01:09<00:00,  5.66it/s]
1 unlabeled	97.821%
2 car   	87.388%
3 person	72.118%
4 bike  	72.445%
5 curve 	38.937%
6 car_stop	34.894%
7 guardrail	49.828%
8 color_cone	59.796%
9 bump  	46.668%
----------     mean_IoU	62.211%	freq_IoU	95.808%	mean_pixel_acc	71.387%	pixel_acc	97.758%
16 03:21:00 Evaluation Elapsed Time: 69.23s


In [22]:

import os

os.chdir("/home/cse-sdpl/Downloads/CMX_E4")

!python eval.py -e 100 -d 0 --split train -p predictions_E4_train

/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 03:21:23 using devices 0
16 03:21:23 Using E4: SegFormer-B2 with reliability-gated rectification
16 03:21:24 Using MLP Decoder
16 03:21:24 Load Model: /home/cse-sdpl/Downloads/CMX_E4/log_TDD_mit_b2_e4/checkpoint/epoch-100.pth
16 03:21:29 Load model, Time usage:
	IO: 4.333745002746582, initialize parameters: 0.04397153854370117
16 03:21:29 GPU 0 handle 1568 data.
  0%|                                                 | 0/1568 [00:00<?, ?it/s][ WARN:0@11.610] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 03:21:33 Save the image 00001D.png
  4%|█▍                                      | 56/1568 [00:13<04:23,  5.74it/s]16 03:21:43 Save 

  8%|██▉                                    | 120/1568 [00:24<04:10,  5.78it/s]16 03:21:54 Save the image 00137D.png


 12%|████▌                                  | 184/1568 [00:35<03:59,  5.78it/s]16 03:22:05 Save the image 00169D.png


 16%|██████▏                                | 248/1568 [00:47<03:53,  5.66it/s]16 03:22:16 Save the image 00201D.png


 20%|███████▊                               | 312/1568 [00:58<03:37,  5.78it/s]16 03:22:27 Save the image 00233D.png


...

 85%|████████████████████████████████▍     | 1336/1568 [03:54<00:40,  5.77it/s]16 03:25:23 Save the image 00886N.png


 89%|█████████████████████████████████▉    | 1400/1568 [04:05<00:29,  5.74it/s]16 03:25:34 Save the image 00918N.png


 93%|███████████████████████████████████▍  | 1464/1568 [04:16<00:17,  5.80it/s]16 03:25:45 Save the image 00950N.png


 97%|█████████████████████████████████████ | 1528/1568 [04:27<00:06,  5.75it/s]16 03:25:56 Save the image 00982N.png


100%|██████████████████████████████████████| 1568/1568 [04:34<00:00,  5.72it/s]
1 unlabeled	99.729%
2 car   	98.371%
3 person	93.343%
4 bike  	94.276%
5 curve 	95.114%
6 car_stop	96.096%
7 guardrail	97.956%
8 color_cone	91.805%
9 bump  	97.413%
----------     mean_IoU	96.011%	freq_IoU	99.483%	mean_pixel_acc	98.149%	pixel_acc	99.738%
16 03:26:03 Evaluation Elapsed Time: 274.06s


# e5

In [23]:
import os

checkpoint = "/home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth"

print("Checkpoint exists:", os.path.exists(checkpoint))

if os.path.exists(checkpoint):
    print(f"Checkpoint size: {os.path.getsize(checkpoint) / (1024**3):.2f} GB")

Checkpoint exists: True
Checkpoint size: 0.75 GB


In [24]:
import os

base = "/home/cse-sdpl/Downloads/CMX_E5"

print("E5 directory exists:", os.path.isdir(base))
print("\nFiles in E5 directory:")

for item in os.listdir(base):
    print(item)

print("\nEvaluation scripts:")
for root, dirs, files in os.walk(base):
    dirs[:] = [d for d in dirs if d not in ["datasets", ".git", "__pycache__"]]
    for file in files:
        if file in ["eval.py", "test.py", "evaluate.py"]:
            print(os.path.join(root, file))

E5 directory exists: True

Files in E5 directory:
config.py
pretraining_backups
.gitignore
eval.py
CMX_Original_Test.ipynb
train.py.pre_e3_init_backup
e1_predictions
CMX_E0_E1_E2_Complete_Comparison.ipynb
train.py
eval.py.repair.bak
utils
config.py.e2_backup
config.py.pre_e5.bak
E2_Learnable_Weighted_Fusion.ipynb
official_checkpoints
eval.ipynb
Untitled.ipynb
pretrained
config.py.pre_e3_backup
results
train.py.e2_backup
train.py.backup
LICENSE
__pycache__
e1_predictions_color
config.py.pre_final_e5.bak
config.py.repair.bak
e5trainig.ipynb
README.md
train.py.pre_e5_init.bak
train.py.before_e4_transfer.bak
config.py.pre_counterfactual.bak
dataloader
engine
.ipynb_checkpoints
E3_Corruption_Aware_Reliability_Fusion.ipynb
segmentation.jpg
train.py.before_e4_audit.bak
CMX_Official_Baseline_Comparison.ipynb
log_TDD_mit_b2_e5
requirements.txt
train.py.pre_optimizer_audit_backup
models
comparison_results

Evaluation scripts:
/home/cse-sdpl/Downloads/CMX_E5/eval.py


In [25]:
cd /home/cse-sdpl/Downloads/CMX_E5

python eval.py --help

grep -nE 'backbone|dataset|checkpoint|num_classes|test|val|split|argparse' eval.py config.py

SyntaxError: invalid syntax (1345769112.py, line 3)

In [26]:

%cd /home/cse-sdpl/Downloads/CMX_E5

!python eval.py --help


/home/cse-sdpl/Downloads/CMX_E5


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
usage: eval.py [-h] [-e EPOCHS] [-d DEVICES] [-v] [--show_image]
               [--save_path SAVE_PATH] [--split {train,val,test}]

options:
  -h, --help            show this help message and exit
  -e EPOCHS, --epochs EPOCHS
  -d DEVICES, --devices DEVICES
  -v, --verbose
  --show_image, -s
  --save_path SAVE_PATH, -p SAVE_PATH
  --split {train,val,test}


In [27]:

!grep -nE 'backbone|dataset|checkpoint|num_classes|test|val|split|argparse' eval.py config.py


eval.py:4:import argparse
eval.py:13:from engine.evaluator import Evaluator
eval.py:22:class SegEvaluator(Evaluator):
eval.py:28:        pred = self.sliding_eval_rgbX(img, modal_x, config.eval_crop_size, config.eval_stride_rate, device)
eval.py:29:        hist_tmp, labeled_tmp, correct_tmp = hist_info(config.num_classes, pred, label)
eval.py:45:            colors = self.dataset.get_class_colors
eval.py:57:        hist = np.zeros((config.num_classes, config.num_classes))
eval.py:69:                                dataset.class_names, show_no_back=False)
eval.py:73:    parser = argparse.ArgumentParser()
eval.py:81:    parser.add_argument('--split', choices=['train', 'val', 'test'], default='val')
eval.py:96:                    'eval_source': config.eval_source,
eval.py:97:                    'test_source': config.test_source,
eval.py:99:    val_pre = ValPre()
eval.py:100:    dataset = RGBXDataset(data_setting, args.split, val_pre)
eval.py:103:        segmentor = SegEvaluator(dataset, con

In [28]:

%cd /home/cse-sdpl/Downloads/CMX_E5

!python eval.py -e 100 -d 0 --split test -p predictions_E5_test


/home/cse-sdpl/Downloads/CMX_E5
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 19:07:47 using devices 0
16 19:07:47 Using E5: SegFormer-B2 with counterfactual reliability
16 19:07:48 Using MLP Decoder
16 19:07:50 Load Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
16 19:07:51 Load model, Time usage:
	IO: 0.751262903213501, initialize parameters: 0.042250633239746094
16 19:07:51 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@7.081] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 19:07:52 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:07<00:35,  

 31%|████████████▍                           | 122/393 [00:13<00:28,  9.67it/s]16 19:08:05 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:20<00:21,  9.65it/s]16 19:08:12 Save the image 01589D.png


 64%|█████████████████████████▋              | 252/393 [00:27<00:14,  9.74it/s]16 19:08:18 Save the image 01236N.png


 81%|████████████████████████████████▎       | 317/393 [00:34<00:07,  9.71it/s]16 19:08:25 Save the image 01301N.png


 97%|██████████████████████████████████████▉ | 382/393 [00:40<00:01,  9.80it/s]16 19:08:32 Save the image 01366N.png


100%|████████████████████████████████████████| 393/393 [00:41<00:00,  9.39it/s]
1 unlabeled	98.318%
2 car   	89.264%
3 person	75.482%
4 bike  	65.006%
5 curve 	47.410%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%
9 bump  	58.436%
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
16 19:08:33 Evaluation Elapsed Time: 41.85s


In [29]:

%cd /home/cse-sdpl/Downloads/CMX_E5

!python eval.py -e 100 -d 0 --split test -p predictions_E5_test


/home/cse-sdpl/Downloads/CMX_E5
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 19:09:43 using devices 0
16 19:09:43 Using E5: SegFormer-B2 with counterfactual reliability
16 19:09:44 Using MLP Decoder
16 19:09:44 Load Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
16 19:09:45 Load model, Time usage:
	IO: 0.7491490840911865, initialize parameters: 0.042465925216674805
16 19:09:45 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@3.650] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 19:09:45 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:06<00:37, 

 31%|████████████▍                           | 122/393 [00:12<00:27,  9.73it/s]16 19:09:58 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:19<00:21,  9.75it/s]16 19:10:04 Save the image 01589D.png


 56%|██████████████████████▍                 | 221/393 [00:23<00:17,  9.92it/s]16 19:10:08 Save the image 01205N.png
16 19:10:08 Save the image 01206N.png
 64%|█████████████████████████▊              | 253/393 [00:26<00:14,  9.87it/s]16 19:10:11 Save the image 01237N.png


 65%|█████████████████████████▉              | 255/393 [00:26<00:13,  9.91it/s]16 19:10:11 Save the image 01239N.png
16 19:10:11 Save the image 01240N.png
 65%|██████████████████████████▏             | 257/393 [00:26<00:13,  9.95it/s]16 19:10:11 Save the image 01241N.png
16 19:10:11 Save the image 01242N.png
 66%|██████████████████████████▎             | 259/393 [00:26<00:13,  9.97it/s]16 19:10:12 Save the image 01243N.png
16 19:10:12 Save the image 01244N.png
 66%|██████████████████████████▌             | 261/393 [00:27<00:13,  9.99it/s]16 19:10:12 Save the image 01245N.png
16 19:10:12 Save the image 01246N.png
 69%|███████████████████████████▊            | 273/393 [00:28<00:12,  9.95it/s]16 19:10:13 Save the image 01257N.png
16 19:10:13 Save the image 01258N.png
 70%|███████████████████████████▉            | 275/393 [00:28<00:11,  9.96it/s]16 19:10:13 Save the image 01259N.png
16 19:10:13 Save the image 01260N.png
 80%|███████████████████████████████▉        | 314/393 [00:32<00:07,  

 98%|███████████████████████████████████████▍| 387/393 [00:39<00:00,  9.92it/s]16 19:10:24 Save the image 01371N.png
16 19:10:25 Save the image 01372N.png


100%|████████████████████████████████████████| 393/393 [00:40<00:00,  9.72it/s]
1 unlabeled	98.318%
2 car   	89.264%
3 person	75.482%
4 bike  	65.006%
5 curve 	47.410%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%
9 bump  	58.436%
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
16 19:10:25 Evaluation Elapsed Time: 40.45s


In [30]:
!python eval.py -e 100 -d 0 --split test -p predictions_E5_test

/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 19:11:51 using devices 0
16 19:11:51 Using E5: SegFormer-B2 with counterfactual reliability
16 19:11:52 Using MLP Decoder
16 19:11:52 Load Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
16 19:11:52 Load model, Time usage:
	IO: 0.7427487373352051, initialize parameters: 0.041689395904541016
16 19:11:52 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@3.599] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 19:11:53 Save the image 01402D.png
 15%|█████▉                                   | 57/393 [00:06<00:34,  9.72it/s]16 19:11:58 Save the i

 31%|████████████▍                           | 122/393 [00:12<00:27,  9.76it/s]16 19:12:05 Save the image 01524D.png


 48%|███████████████████                     | 187/393 [00:19<00:21,  9.73it/s]16 19:12:12 Save the image 01589D.png


 54%|█████████████████████▊                  | 214/393 [00:22<00:18,  9.83it/s]16 19:12:15 Save the image 01198N.png
16 19:12:15 Save the image 01199N.png
 61%|████████████████████████▍               | 240/393 [00:24<00:15,  9.92it/s]16 19:12:17 Save the image 01224N.png
16 19:12:17 Save the image 01225N.png
 64%|█████████████████████████▊              | 253/393 [00:26<00:14,  9.87it/s]16 19:12:18 Save the image 01237N.png


 65%|██████████████████████████              | 256/393 [00:26<00:13,  9.90it/s]16 19:12:19 Save the image 01240N.png
16 19:12:19 Save the image 01241N.png
 81%|████████████████████████████████▍       | 319/393 [00:32<00:07,  9.67it/s]16 19:12:25 Save the image 01303N.png


 98%|███████████████████████████████████████ | 384/393 [00:39<00:00,  9.91it/s]16 19:12:32 Save the image 01368N.png


100%|████████████████████████████████████████| 393/393 [00:40<00:00,  9.68it/s]
1 unlabeled	98.318%
2 car   	89.264%
3 person	75.482%
4 bike  	65.006%
5 curve 	47.410%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%
9 bump  	58.436%
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
16 19:12:33 Evaluation Elapsed Time: 40.60s


In [31]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!grep -n -E 'split|test.txt|val.txt|test' eval.py

/home/cse-sdpl/Downloads/CMX_E5
81:    parser.add_argument('--split', choices=['train', 'val', 'test'], default='val')
97:                    'test_source': config.test_source,
100:    dataset = RGBXDataset(data_setting, args.split, val_pre)


In [32]:
!ls -ld predictions_E5_val predictions_E5_test
!find predictions_E5_val -type f | wc -l
!find predictions_E5_test -type f | wc -l

ls: cannot access 'predictions_E5_val': No such file or directory
drwxrwxr-x 2 cse-sdpl cse-sdpl 12288 Oct 16 19:08 predictions_E5_test
find: ‘predictions_E5_val’: No such file or directory
0
393


In [33]:
%cd /home/cse-sdpl/Downloads/CMX_E5

# Check dataset split paths configured in the project
!grep -n -E 'val_source|test_source|train_source' config.py

# Inspect the saved test predictions
!ls -lh predictions_E5_test | head -15

# Check whether validation predictions exist under another name
!find . -maxdepth 2 -type d -iname '*val*' | head -30

/home/cse-sdpl/Downloads/CMX_E5
43:C.train_source = osp.join(C.dataset_path, 'train.txt')
44:C.eval_source = osp.join(C.dataset_path, 'val.txt')
45:C.test_source = osp.join(C.dataset_path, 'test.txt')
total 1.7M
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.7K Oct 16 19:12 01189N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.0K Oct 16 19:12 01190N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 1.2K Oct 16 19:12 01191N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 1.2K Oct 16 19:12 01192N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 1.2K Oct 16 19:12 01193N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.0K Oct 16 19:12 01194N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.6K Oct 16 19:12 01195N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 1.7K Oct 16 19:12 01196N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.9K Oct 16 19:12 01197N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.2K Oct 16 19:12 01198N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.3K Oct 16 19:12 01199N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.1K Oct 16 19:12 01200N.png
-rw-rw-r-- 1 cse-sdpl cse-sdpl 2.0K Oct 16 19:12 01201N.png
-rw-rw-r

In [34]:

%cd /home/cse-sdpl/Downloads/CMX_E5

# 1. Check how eval.py chooses the split and loads the dataset
!sed -n '70,125p' eval.py

# 2. Check whether val_pre changes the split behavior
!grep -n -E 'val_pre|eval_source|test_source|RGBXDataset' eval.py

# 3. Check the list files and their first entries
!head -5 /home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/test.txt
!head -5 /home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/val.txt


/home/cse-sdpl/Downloads/CMX_E5
        return result_line

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('-e', '--epochs', default='last', type=str)
    parser.add_argument('-d', '--devices', default='0', type=str)
    parser.add_argument('-v', '--verbose', default=False, action='store_true')
    parser.add_argument('--show_image', '-s', default=False,
                        action='store_true')
    parser.add_argument('--save_path', '-p', default=None)

    parser.add_argument('--split', choices=['train', 'val', 'test'], default='val')
    args = parser.parse_args()
    all_dev = parse_devices(args.devices)

    network = segmodel(cfg=config, criterion=None, norm_layer=nn.BatchNorm2d)
    data_setting = {'rgb_root': config.rgb_root_folder,
                    'rgb_format': config.rgb_format,
                    'gt_root': config.gt_root_folder,
                    'gt_format': config.gt_format,
                    'transform_gt': config.gt

In [35]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!grep -n -A45 -B10 'def __init__' dataloader/RGBXDataset.py

/home/cse-sdpl/Downloads/CMX_E5
1-import os
2-from pickletools import uint8
3-import cv2
4-import torch
5-import numpy as np
6-
7-import torch.utils.data as data
8-
9-
10-class RGBXDataset(data.Dataset):
11:    def __init__(self, setting, split_name, preprocess=None, file_length=None):
12-        super(RGBXDataset, self).__init__()
13-        self._split_name = split_name
14-        self._rgb_path = setting['rgb_root']
15-        self._rgb_format = setting['rgb_format']
16-        self._gt_path = setting['gt_root']
17-        self._gt_format = setting['gt_format']
18-        self._transform_gt = setting['transform_gt']
19-        self._x_path = setting['x_root']
20-        self._x_format = setting['x_format']
21-        self._x_single_channel = setting['x_single_channel']
22-        self._train_source = setting['train_source']
23-        self._eval_source = setting['eval_source']
24-        self._test_source = setting['test_source']
25-        self.class_names = setting['class_names']


In [36]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!grep -n -A30 -B5 'def _get_file_names' dataloader/RGBXDataset.py

/home/cse-sdpl/Downloads/CMX_E5
60-
61-        output_dict = dict(data=rgb, label=gt, modal_x=x, fn=str(item_name), n=len(self._file_names))
62-
63-        return output_dict
64-
65:    def _get_file_names(self, split_name):
66-        assert split_name in ['train', 'val', 'test']
67-        source = self._train_source
68-        if split_name == "val":
69-            source = self._eval_source
70-        elif split_name == "test":
71-            source = self._test_source
72-
73-        file_names = []
74-        with open(source) as f:
75-            files = f.readlines()
76-
77-        for item in files:
78-            file_name = item.strip()
79-            file_names.append(file_name)
80-
81-        return file_names
82-
83-    def _construct_new_file_names(self, length):
84-        assert isinstance(length, int)
85-        files_len = len(self._file_names)                          
86-        new_file_names = self._file_names * (length // files_len)   
87-
88-        rand_indices

In [37]:
import config
from dataloader.RGBXDataset import RGBXDataset
from dataloader.RGBXDataset import ValPre

setting = {
    'rgb_root': config.rgb_root_folder,
    'rgb_format': config.rgb_format,
    'gt_root': config.gt_root_folder,
    'gt_format': config.gt_format,
    'transform_gt': config.gt_transform,
    'x_root': config.x_root_folder,
    'x_format': config.x_format,
    'x_single_channel': config.x_is_single_channel,
    'class_names': config.class_names,
    'train_source': config.train_source,
    'eval_source': config.eval_source,
    'test_source': config.test_source,
}

for split in ['val', 'test']:
    ds = RGBXDataset(setting, split, ValPre())
    print(f"{split}: {len(ds)} images")
    print(f"First 5 IDs: {ds._file_names[:5]}")

ImportError: cannot import name 'ValPre' from 'dataloader.RGBXDataset' (/home/cse-sdpl/Downloads/CMX_E5/dataloader/RGBXDataset.py)

In [38]:
%cd /home/cse-sdpl/Downloads/CMX_E5

import config
from dataloader.RGBXDataset import RGBXDataset

setting = {
    'rgb_root': config.rgb_root_folder,
    'rgb_format': config.rgb_format,
    'gt_root': config.gt_root_folder,
    'gt_format': config.gt_format,
    'transform_gt': config.gt_transform,
    'x_root': config.x_root_folder,
    'x_format': config.x_format,
    'x_single_channel': config.x_is_single_channel,
    'class_names': config.class_names,
    'train_source': config.train_source,
    'eval_source': config.eval_source,
    'test_source': config.test_source,
}

for split in ['val', 'test']:
    ds = RGBXDataset(setting, split, preprocess=None)
    print(f"\n{split.upper()} images: {len(ds)}")
    print("First 5 IDs:", ds._file_names[:5])
    print("Last 5 IDs:", ds._file_names[-5:])

/home/cse-sdpl/Downloads/CMX_E5


/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


AttributeError: module 'config' has no attribute 'rgb_root_folder'

In [39]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!head -70 eval.py
!grep -n -E 'rgb_root_folder|gt_root_folder|x_root_folder|class_names|config =' config.py
!grep -n -A25 'def _get_file_names' dataloader/RGBXDataset.py

/home/cse-sdpl/Downloads/CMX_E5
from PIL import Image
import os
import cv2
import argparse
import numpy as np

import torch
import torch.nn as nn

from config import config
from utils.pyt_utils import ensure_dir, link_file, load_model, parse_devices
from utils.visualize import print_iou, show_img
from engine.evaluator import Evaluator
from engine.logger import get_logger
from utils.metric import hist_info, compute_score
from dataloader.RGBXDataset import RGBXDataset
from models.builder import EncoderDecoder as segmodel
from dataloader.dataloader import ValPre

logger = get_logger()

class SegEvaluator(Evaluator):
    def func_per_iteration(self, data, device):
        img = data['data']
        label = data['label']
        modal_x = data['modal_x']
        name = data['fn']
        pred = self.sliding_eval_rgbX(img, modal_x, config.eval_crop_size, config.eval_stride_rate, device)
        hist_tmp, labeled_tmp, correct_tmp = hist_info(config.num_classes, pred, label)
        results_di

In [40]:
%cd /home/cse-sdpl/Downloads/CMX_E5

from config import config
from dataloader.RGBXDataset import RGBXDataset

setting = {
    'rgb_root': config.rgb_root_folder,
    'rgb_format': config.rgb_format,
    'gt_root': config.gt_root_folder,
    'gt_format': config.gt_format,
    'transform_gt': config.gt_transform,
    'x_root': config.x_root_folder,
    'x_format': config.x_format,
    'x_single_channel': config.x_is_single_channel,
    'class_names': config.class_names,
    'train_source': config.train_source,
    'eval_source': config.eval_source,
    'test_source': config.test_source,
}

for split in ['val', 'test']:
    ds = RGBXDataset(setting, split, preprocess=None)
    print(f"\n{split.upper()} count: {len(ds)}")
    print("First 5 IDs:", ds._file_names[:5])
    print("Last 5 IDs:", ds._file_names[-5:])

/home/cse-sdpl/Downloads/CMX_E5

VAL count: 392
First 5 IDs: ['00498D', '00499D', '00500D', '00501D', '00502D']
Last 5 IDs: ['01184N', '01185N', '01186N', '01187N', '01188N']

TEST count: 393
First 5 IDs: ['01402D', '01403D', '01404D', '01405D', '01406D']
Last 5 IDs: ['01372N', '01373N', '01374N', '01375N', '01376N']


In [41]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!grep -R -n -E 'mean_IoU|freq_IoU|58.379|98.318' log_TDD_mit_b2_e5 2>/dev/null | tail -30


/home/cse-sdpl/Downloads/CMX_E5
log_TDD_mit_b2_e5/val_2026_10_16_19_11_50.log:2:1 unlabeled	98.318%
log_TDD_mit_b2_e5/val_2026_10_16_19_11_50.log:11:----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
log_TDD_mit_b2_e5/val_2026_10_16_19_09_42.log:2:1 unlabeled	98.318%
log_TDD_mit_b2_e5/val_2026_10_16_19_09_42.log:11:----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
log_TDD_mit_b2_e5/val_last.log:2:1 unlabeled	98.318%
log_TDD_mit_b2_e5/val_last.log:11:----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
log_TDD_mit_b2_e5/val_2026_10_16_19_07_46.log:2:1 unlabeled	98.318%
log_TDD_mit_b2_e5/val_2026_10_16_19_07_46.log:11:----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%


In [42]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!grep -n -A100 -B15 'def run' engine/evaluator.py

/home/cse-sdpl/Downloads/CMX_E5
27-        self.multi_scales = multi_scales
28-        self.is_flip = is_flip
29-        self.network = network
30-        self.devices = devices
31-
32-        self.context = mp.get_context('spawn')
33-        self.val_func = None
34-        self.results_queue = self.context.Queue(self.ndata)
35-
36-        self.verbose = verbose
37-        self.save_path = save_path
38-        if save_path is not None:
39-            ensure_dir(save_path)
40-        self.show_image = show_image
41-
42:    def run(self, model_path, model_indice, log_file, log_file_link):
43-        """There are four evaluation modes:
44-            1.only eval a .pth model: -e *.pth
45-            2.only eval a certain epoch: -e epoch
46-            3.eval all epochs in a given section: -e start_epoch-end_epoch
47-            4.eval all epochs from a certain started epoch: -e start_epoch-
48-            """
49-        if '.pth' in model_indice:
50-            models = [model_indice, ]
5

In [43]:
from pathlib import Path

test_file = Path(
    "/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/test.txt"
)
pred_dir = Path("predictions_E5_test")

test_ids = {line.strip() for line in test_file.read_text().splitlines() if line.strip()}
pred_ids = {p.stem for p in pred_dir.glob("*.png")}

print("Test IDs:", len(test_ids))
print("Prediction IDs:", len(pred_ids))
print("Predictions matching test IDs:", len(test_ids & pred_ids))
print("Predictions not in test list:", sorted(pred_ids - test_ids)[:10])
print("Missing test predictions:", sorted(test_ids - pred_ids)[:10])

Test IDs: 393
Prediction IDs: 393
Predictions matching test IDs: 393
Predictions not in test list: []
Missing test predictions: []


In [44]:
%cd /home/cse-sdpl/Downloads/CMX_E5

from config import config
from dataloader.RGBXDataset import RGBXDataset
from dataloader.dataloader import ValPre

setting = {
    'rgb_root': config.rgb_root_folder,
    'rgb_format': config.rgb_format,
    'gt_root': config.gt_root_folder,
    'gt_format': config.gt_format,
    'transform_gt': config.gt_transform,
    'x_root': config.x_root_folder,
    'x_format': config.x_format,
    'x_single_channel': config.x_is_single_channel,
    'class_names': config.class_names,
    'train_source': config.train_source,
    'eval_source': config.eval_source,
    'test_source': config.test_source,
}

test_ds = RGBXDataset(setting, 'test', ValPre())

print("Dataset length:", len(test_ds))
for i in [0, 1, 2, len(test_ds)-1]:
    item = test_ds[i]
    print(i, item['fn'])

/home/cse-sdpl/Downloads/CMX_E5
Dataset length: 393
0 01402D
1 01403D
2 01404D
392 01376N


In [45]:
%cd /home/cse-sdpl/Downloads/CMX_E5

from pathlib import Path
import hashlib

folder = Path("predictions_E5_test")
files = sorted(folder.glob("*.png"))

print("Number of prediction files:", len(files))

for f in files[:5]:
    digest = hashlib.md5(f.read_bytes()).hexdigest()
    print(f.name, digest)

/home/cse-sdpl/Downloads/CMX_E5
Number of prediction files: 393
01189N.png e4ee2c345e00d55250236dfcbc2f52ef
01190N.png 47270e7d8b1398f251253d1f2dd71eb4
01191N.png a50689866c60ea907ad9ce265c0f10ab
01192N.png a50689866c60ea907ad9ce265c0f10ab
01193N.png a50689866c60ea907ad9ce265c0f10ab


In [46]:
from pathlib import Path

base = Path("/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD")
pred_dir = Path("/home/cse-sdpl/Downloads/CMX_E5/predictions_E5_test")

def read_ids(path):
    return {x.strip() for x in path.read_text().splitlines() if x.strip()}

val_ids = read_ids(base / "val.txt")
test_ids = read_ids(base / "test.txt")
pred_ids = {p.stem for p in pred_dir.glob("*.png")}

print("Predictions matching validation IDs:", len(pred_ids & val_ids))
print("Predictions matching test IDs:", len(pred_ids & test_ids))
print("First 10 prediction IDs:", sorted(pred_ids)[:10])
print("First 10 test IDs:", sorted(test_ids)[:10])

Predictions matching validation IDs: 0
Predictions matching test IDs: 393
First 10 prediction IDs: ['01189N', '01190N', '01191N', '01192N', '01193N', '01194N', '01195N', '01196N', '01197N', '01198N']
First 10 test IDs: ['01189N', '01190N', '01191N', '01192N', '01193N', '01194N', '01195N', '01196N', '01197N', '01198N']


In [47]:
from pathlib import Path
import numpy as np
from PIL import Image

base = Path("/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD")
project = Path("/home/cse-sdpl/Downloads/CMX_E5")

val_ids = {x.strip() for x in (base / "val.txt").read_text().splitlines() if x.strip()}
test_ids = {x.strip() for x in (base / "test.txt").read_text().splitlines() if x.strip()}

print("Validation/test ID intersection:", len(val_ids & test_ids))
print("Test prediction count:", len(list((project / "predictions_E5_test").glob("*.png"))))

Validation/test ID intersection: 0
Test prediction count: 393


In [48]:
from pathlib import Path
import hashlib

base = Path("/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD")

print("Label root:", base / "labels")

for split in ["val", "test"]:
    ids = [
        x.strip()
        for x in (base / f"{split}.txt").read_text().splitlines()
        if x.strip()
    ]

    print(f"\n{split.upper()} — {len(ids)} images")
    for image_id in ids[:3]:
        matches = list(base.rglob(image_id + ".png"))
        print(image_id, [str(p) for p in matches[:3]])

Label root: /home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/labels

VAL — 392 images
00498D ['/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/RGB/00498D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Thermal/00498D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Label/00498D.png']
00499D ['/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/RGB/00499D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Thermal/00499D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Label/00499D.png']
00500D ['/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/RGB/00500D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Thermal/00500D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Label/00500D.png']

TEST — 393 images
01402D ['/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/RGB/01402D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Thermal/01402D.png', '/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD/Label/01402D.png']
01403D ['

In [49]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!python eval.py -e 100 -d 0 --split test -p predictions_E5_test_verify

/home/cse-sdpl/Downloads/CMX_E5
/home/cse-sdpl/miniconda3/envs/cmx/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
16 19:23:30 using devices 0
16 19:23:30 Using E5: SegFormer-B2 with counterfactual reliability
16 19:23:31 Using MLP Decoder
16 19:23:34 Load Model: /home/cse-sdpl/Downloads/CMX_E5/log_TDD_mit_b2_e5/checkpoint/epoch-100.pth
16 19:23:34 Load model, Time usage:
	IO: 0.7498226165771484, initialize parameters: 0.04236721992492676
16 19:23:34 GPU 0 handle 393 data.
  0%|                                                  | 0/393 [00:00<?, ?it/s][ WARN:0@7.604] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
16 19:23:36 Save the image 01402D.png
  1%|▍                                         | 4/393 [00:01<01:43,  

 31%|████████████▌                           | 123/393 [00:13<00:28,  9.57it/s]16 19:23:48 Save the image 01525D.png


 48%|███████████████████▏                    | 188/393 [00:20<00:20,  9.84it/s]16 19:23:55 Save the image 01590D.png


 52%|████████████████████▊                   | 205/393 [00:22<00:19,  9.86it/s]16 19:23:57 Save the image 01189N.png
16 19:23:57 Save the image 01190N.png
 53%|█████████████████████▏                  | 208/393 [00:22<00:18,  9.95it/s]16 19:23:57 Save the image 01192N.png
16 19:23:57 Save the image 01193N.png
 54%|█████████████████████▊                  | 214/393 [00:23<00:18,  9.92it/s]16 19:23:58 Save the image 01198N.png
16 19:23:58 Save the image 01199N.png
 55%|█████████████████████▉                  | 216/393 [00:23<00:17,  9.97it/s]16 19:23:58 Save the image 01200N.png
16 19:23:58 Save the image 01201N.png
 55%|██████████████████████▏                 | 218/393 [00:23<00:17,  9.99it/s]16 19:23:58 Save the image 01202N.png
16 19:23:58 Save the image 01203N.png
 56%|██████████████████████▍                 | 221/393 [00:23<00:17,  9.98it/s]16 19:23:58 Save the image 01205N.png
16 19:23:59 Save the image 01206N.png
 58%|███████████████████████                 | 226/393 [00:24<00:16,  

16 19:24:03 Save the image 01252N.png
 69%|███████████████████████████▍            | 270/393 [00:28<00:12,  9.99it/s]16 19:24:03 Save the image 01254N.png
16 19:24:03 Save the image 01255N.png
 69%|███████████████████████████▋            | 272/393 [00:28<00:12, 10.00it/s]16 19:24:04 Save the image 01256N.png
16 19:24:04 Save the image 01257N.png
 70%|███████████████████████████▉            | 274/393 [00:29<00:11, 10.01it/s]16 19:24:04 Save the image 01258N.png
16 19:24:04 Save the image 01259N.png
 70%|████████████████████████████            | 276/393 [00:29<00:11, 10.02it/s]16 19:24:04 Save the image 01260N.png
16 19:24:04 Save the image 01261N.png
 71%|████████████████████████████▎           | 278/393 [00:29<00:11, 10.01it/s]16 19:24:04 Save the image 01262N.png
16 19:24:04 Save the image 01263N.png
 71%|████████████████████████████▍           | 280/393 [00:29<00:11, 10.00it/s]16 19:24:04 Save the image 01264N.png
16 19:24:04 Save the image 01265N.png
 72%|███████████████████████████

100%|████████████████████████████████████████| 393/393 [00:41<00:00,  9.48it/s]
1 unlabeled	98.318%
2 car   	89.264%
3 person	75.482%
4 bike  	65.006%
5 curve 	47.410%
6 car_stop	32.014%
7 guardrail	7.243%
8 color_cone	52.236%
9 bump  	58.436%
----------     mean_IoU	58.379%	freq_IoU	96.795%	mean_pixel_acc	67.820%	pixel_acc	98.250%
16 19:24:16 Evaluation Elapsed Time: 41.44s


In [50]:
from pathlib import Path
from PIL import Image
import numpy as np

folder = Path("predictions_E5_test_verify")
files = sorted(folder.glob("*.png"))

print("Test prediction count:", len(files))
print("First 10 prediction files:", [p.name for p in files[:10]])

for p in files[:5]:
    arr = np.array(Image.open(p))
    print(
        p.name,
        "shape:", arr.shape,
        "unique predicted values:", np.unique(arr)[:20],
        "unique-value count:", len(np.unique(arr))
    )

Test prediction count: 393
First 10 prediction files: ['01189N.png', '01190N.png', '01191N.png', '01192N.png', '01193N.png', '01194N.png', '01195N.png', '01196N.png', '01197N.png', '01198N.png']
01189N.png shape: (480, 640) unique predicted values: [0 2 3] unique-value count: 3
01190N.png shape: (480, 640) unique predicted values: [0 2 3] unique-value count: 3
01191N.png shape: (480, 640) unique predicted values: [0] unique-value count: 1
01192N.png shape: (480, 640) unique predicted values: [0] unique-value count: 1
01193N.png shape: (480, 640) unique predicted values: [0] unique-value count: 1


In [51]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!grep -n -A35 -B8 "def __getitem__" dataloader/RGBXDataset.py
!grep -n -A35 -B8 "def func_per_iteration" eval.py
!grep -n -A35 -B8 "def run" eval.py

/home/cse-sdpl/Downloads/CMX_E5
27-        self._file_length = file_length
28-        self.preprocess = preprocess
29-
30-    def __len__(self):
31-        if self._file_length is not None:
32-            return self._file_length
33-        return len(self._file_names)
34-
35:    def __getitem__(self, index):
36-        if self._file_length is not None:
37-            item_name = self._construct_new_file_names(self._file_length)[index]
38-        else:
39-            item_name = self._file_names[index]
40-        rgb_path = os.path.join(self._rgb_path, item_name + self._rgb_format)
41-        x_path = os.path.join(self._x_path, item_name + self._x_format)
42-        gt_path = os.path.join(self._gt_path, item_name + self._gt_format)
43-
44-        # Check the following settings if necessary
45-        rgb = self._open_image(rgb_path, cv2.COLOR_BGR2RGB)
46-
47-        gt = self._open_image(gt_path, cv2.IMREAD_GRAYSCALE, dtype=np.uint8)
48-        if self._transform_gt:
49-            gt 

In [52]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!grep -n -A70 -B10 "def main" eval.py
!grep -n -A35 -B5 "def __init__" dataloader/RGBXDataset.py
!grep -n -A45 -B5 "def compute_metric" eval.py
!grep -n -A35 -B5 "def run" eval.py

/home/cse-sdpl/Downloads/CMX_E5
6-
7-import torch.utils.data as data
8-
9-
10-class RGBXDataset(data.Dataset):
11:    def __init__(self, setting, split_name, preprocess=None, file_length=None):
12-        super(RGBXDataset, self).__init__()
13-        self._split_name = split_name
14-        self._rgb_path = setting['rgb_root']
15-        self._rgb_format = setting['rgb_format']
16-        self._gt_path = setting['gt_root']
17-        self._gt_format = setting['gt_format']
18-        self._transform_gt = setting['transform_gt']
19-        self._x_path = setting['x_root']
20-        self._x_format = setting['x_format']
21-        self._x_single_channel = setting['x_single_channel']
22-        self._train_source = setting['train_source']
23-        self._eval_source = setting['eval_source']
24-        self._test_source = setting['test_source']
25-        self.class_names = setting['class_names']
26-        self._file_names = self._get_file_names(split_name)
27-        self._file_length =

In [53]:
%cd /home/cse-sdpl/Downloads/CMX_E5

!sed -n '99,165p' eval.py
!sed -n '65,105p' dataloader/RGBXDataset.py

/home/cse-sdpl/Downloads/CMX_E5
    val_pre = ValPre()
    dataset = RGBXDataset(data_setting, args.split, val_pre)
 
    with torch.no_grad():
        segmentor = SegEvaluator(dataset, config.num_classes, config.norm_mean,
                                 config.norm_std, network,
                                 config.eval_scale_array, config.eval_flip,
                                 all_dev, args.verbose, args.save_path,
                                 args.show_image)
        segmentor.run(config.checkpoint_dir, args.epochs, config.val_log_file,
                      config.link_val_log_file)
    def _get_file_names(self, split_name):
        assert split_name in ['train', 'val', 'test']
        source = self._train_source
        if split_name == "val":
            source = self._eval_source
        elif split_name == "test":
            source = self._test_source

        file_names = []
        with open(source) as f:
            files = f.readlines()

        for item in fi

In [54]:
from collections import Counter
from pathlib import Path
import cv2
import numpy as np

root = Path("/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD")
label_dir = root / "Label"

for split in ["val", "test"]:
    ids = [
        x.strip()
        for x in (root / f"{split}.txt").read_text().splitlines()
        if x.strip()
    ]

    counts = Counter()
    for image_id in ids:
        label = cv2.imread(str(label_dir / f"{image_id}.png"),
                           cv2.IMREAD_GRAYSCALE)
        if label is not None:
            values, frequencies = np.unique(label, return_counts=True)
            counts.update(dict(zip(values.tolist(), frequencies.tolist())))

    print(f"\n{split.upper()} — {len(ids)} images")
    print("Raw label pixel counts:", dict(sorted(counts.items())))


VAL — 392 images
Raw label pixel counts: {0: 110824522, 1: 4219671, 2: 1106006, 3: 2122305, 4: 684884, 5: 499275, 6: 75317, 7: 361079, 8: 529341}

TEST — 393 images
Raw label pixel counts: {0: 112023545, 1: 4854504, 2: 1345972, 3: 1261567, 4: 602620, 5: 355314, 6: 17794, 7: 85246, 8: 183038}


In [55]:

from pathlib import Path
from collections import Counter
import cv2
import numpy as np

root = Path("/home/cse-sdpl/Downloads/CMX_Original/datasets/TDD")
pred_dir = Path("/home/cse-sdpl/Downloads/CMX_E5/predictions_E5_test_verify")

ids = [
    x.strip()
    for x in (root / "test.txt").read_text().splitlines()
    if x.strip()
]

hist = np.zeros((9, 9), dtype=np.int64)
missing = []
shape_errors = []

for image_id in ids:
    gt_path = root / "Label" / f"{image_id}.png"
    pred_path = pred_dir / f"{image_id}.png"

    gt = cv2.imread(str(gt_path), cv2.IMREAD_GRAYSCALE)
    pred = cv2.imread(str(pred_path), cv2.IMREAD_GRAYSCALE)

    if gt is None or pred is None:
        missing.append(image_id)
        continue

    if gt.shape != pred.shape:
        shape_errors.append((image_id, gt.shape, pred.shape))
        continue

    valid = (gt >= 0) & (gt < 9) & (pred >= 0) & (pred < 9)
    encoded = 9 * gt[valid].astype(np.int64) + pred[valid].astype(np.int64)
    hist += np.bincount(encoded, minlength=81).reshape(9, 9)

tp = np.diag(hist)
gt_count = hist.sum(axis=1)
pred_count = hist.sum(axis=0)
union = gt_count + pred_count - tp

iou = np.divide(tp, union, out=np.zeros(9, dtype=float), where=union > 0)
class_acc = np.divide(tp, gt_count, out=np.zeros(9, dtype=float), where=gt_count > 0)

print("Test IDs:", len(ids))
print("Missing files:", len(missing))
print("Shape errors:", len(shape_errors))
print("Confusion matrix:\n", hist)
print("Class IoU (%):", np.round(iou * 100, 3))
print("Mean IoU (%):", round(iou[union > 0].mean() * 100, 3))
print("Mean class accuracy (%):", round(class_acc[gt_count > 0].mean() * 100, 3))
print("Pixel accuracy (%):", round(tp.sum() / max(hist.sum(), 1) * 100, 3))


Test IDs: 393
Missing files: 0
Shape errors: 0
Confusion matrix:
 [[111288758    215373    114731    148646    185341     23516      5034
       4496     37650]
 [   260457   4569084     24277       589         3        55         2
         37         0]
 [   161101     26671   1136966     19791        80      1327        36
          0         0]
 [   283073     16994     17905    939566        21      3278         0
        730         0]
 [   226138        21        45      1759    374024       263         0
        370         0]
 [   148427      3958      2902     12366       208    126227     61226
          0         0]
 [    11383         0       280         0         0        40      6091
          0         0]
 [    25284      1055       159       630       150     10496         0
      47472         0]
 [    53561        23         0         0       492         0         0
          0    128962]]
Class IoU (%): [98.318 89.264 75.482 65.006 47.41  32.014  7.243 52.236 58.436

# plot and graphs

In [57]:

# ============================================================
# CMX E0-E5: COMPLETE MODEL COMPARISON PIPELINE
# Run this cell in Jupyter on your Ubuntu machine
# ============================================================

from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import os
import sys
import json
import time

# -------------------- CONFIGURATION -------------------------

BASE = Path("/home/cse-sdpl/Downloads")

PROJECTS = {
    "E0": BASE / "CMX_Original",
    "E1": BASE / "CMX_E1",
    "E2": BASE / "CMX_E2",
    "E3": BASE / "CMX_E3",
    "E4": BASE / "CMX_E4",
    "E5": BASE / "CMX_E5",
}

# Change this if your Conda environment has a different Python.
PYTHON = sys.executable

# Use the same epoch/checkpoint selection rule for every model.
EPOCH = 100
DEVICE = "0"

# Your CMX evaluation script's split names.
SPLITS = ["train", "val", "test"]

OUT = BASE / "CMX_ALL_MODEL_COMPARISON"
OUT.mkdir(parents=True, exist_ok=True)

for split in SPLITS:
    (OUT / split).mkdir(exist_ok=True)

# -------------------- FIND CHECKPOINTS ----------------------

def find_checkpoint(project):
    checkpoint_dir = project / "log_TDD_mit_b2" / "checkpoint"

    # Prefer the epoch-100 checkpoint.
    candidates = [
        checkpoint_dir / f"epoch-{EPOCH}.pth",
        checkpoint_dir / f"epoch-{EPOCH}.pth.tar",
        project / "checkpoint" / f"epoch-{EPOCH}.pth",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    # Search recursively if the log directory differs.
    found = list(project.rglob(f"epoch-{EPOCH}.pth"))
    if found:
        return found[0]

    # Fallback to latest/last checkpoint if epoch 100 is absent.
    for pattern in ["epoch-last.pth", "*best*.pth", "*.pth"]:
        found = sorted(project.rglob(pattern))
        if found:
            return found[-1]

    return None


# -------------------- METRIC PARSER -------------------------

def extract_metrics(text):
    """
    Parse the metric names commonly printed by CMX eval.py.
    Missing metrics remain NaN; they are never set to zero.
    """
    patterns = {
        "mIoU": [
            r"mean_IoU\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
            r"Mean IoU\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
        ],
        "FWIoU": [
            r"freq_IoU\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
            r"Frequency.weighted IoU\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
        ],
        "mPA": [
            r"mean_pixel_acc\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
            r"Mean Pixel Accuracy\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
        ],
        "PA": [
            r"pixel_acc\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
            r"Pixel Accuracy\s*[:=]?\s*([0-9]+\.?[0-9]*)\s*%?",
        ],
    }

    result = {}
    for metric, regexes in patterns.items():
        result[metric] = np.nan
        for pattern in regexes:
            matches = re.findall(pattern, text, flags=re.IGNORECASE)
            if matches:
                result[metric] = float(matches[-1])
                break

    # Detect class-wise lines such as:
    # 1 unlabeled 97.821%
    # 2 car 87.388%
    class_rows = []
    for line in text.splitlines():
        match = re.match(
            r"^\s*\d+\s+([A-Za-z_][A-Za-z_0-9 -]*?)\s+"
            r"([0-9]+\.?[0-9]*)\s*%\s*$",
            line
        )
        if match:
            class_rows.append({
                "Class": match.group(1).strip(),
                "IoU": float(match.group(2))
            })

    result["ClassIoU"] = class_rows
    return result


# -------------------- RUN EVALUATION ------------------------

all_results = []
all_class_results = []
run_status = []

for model_name, project in PROJECTS.items():

    print("\n" + "=" * 72)
    print(f"MODEL: {model_name} | PROJECT: {project}")
    print("=" * 72)

    if not project.exists():
        print("SKIPPED: project folder does not exist.")
        run_status.append({
            "Model": model_name,
            "Split": "all",
            "Status": "Project folder missing",
            "Project": str(project),
        })
        continue

    eval_script = project / "eval.py"
    if not eval_script.exists():
        print("SKIPPED: eval.py not found.")
        run_status.append({
            "Model": model_name,
            "Split": "all",
            "Status": "eval.py missing",
            "Project": str(project),
        })
        continue

    checkpoint = find_checkpoint(project)
    if checkpoint is None:
        print("SKIPPED: no checkpoint found.")
        run_status.append({
            "Model": model_name,
            "Split": "all",
            "Status": "Checkpoint missing",
            "Project": str(project),
        })
        continue

    print("Checkpoint:", checkpoint)

    for split in SPLITS:
        print(f"\nEvaluating {model_name} on {split}...")

        prediction_dir = OUT / split / f"predictions_{model_name}"
        prediction_dir.mkdir(parents=True, exist_ok=True)

        log_path = OUT / split / f"{model_name}_{split}.log"

        # Each subprocess runs inside its own project directory,
        # so its local config.py and models/builder.py are used.
        command = [
            PYTHON,
            str(eval_script),
            "-e", str(EPOCH),
            "-d", DEVICE,
            "--split", split,
            "-p", str(prediction_dir),
        ]

        try:
            completed = subprocess.run(
                command,
                cwd=str(project),
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                timeout=None,
            )

            log_text = completed.stdout or ""
            log_path.write_text(log_text, encoding="utf-8")

            if completed.returncode != 0:
                print(f"FAILED. Check log: {log_path}")
                print(log_text[-2500:])
                run_status.append({
                    "Model": model_name,
                    "Split": split,
                    "Status": f"Evaluation failed (exit {completed.returncode})",
                    "Project": str(project),
                })
                continue

            metrics = extract_metrics(log_text)

            row = {
                "Model": model_name,
                "Split": split,
                "Checkpoint": str(checkpoint),
                "mIoU": metrics["mIoU"],
                "PA": metrics["PA"],
                "mPA": metrics["mPA"],
                "FWIoU": metrics["FWIoU"],
                "Log": str(log_path),
            }
            all_results.append(row)

            for item in metrics["ClassIoU"]:
                all_class_results.append({
                    "Model": model_name,
                    "Split": split,
                    "Class": item["Class"],
                    "IoU": item["IoU"],
                })

            run_status.append({
                "Model": model_name,
                "Split": split,
                "Status": "Completed; metrics parsed",
                "Project": str(project),
            })

            print("mIoU:", metrics["mIoU"])
            print("PA:", metrics["PA"])
            print("mPA:", metrics["mPA"])
            print("FWIoU:", metrics["FWIoU"])
            print("Saved log:", log_path)

        except Exception as exc:
            print("ERROR:", repr(exc))
            run_status.append({
                "Model": model_name,
                "Split": split,
                "Status": f"Error: {exc}",
                "Project": str(project),
            })


# -------------------- SAVE TABLES ---------------------------

results_df = pd.DataFrame(all_results)
class_df = pd.DataFrame(all_class_results)
status_df = pd.DataFrame(run_status)

results_df.to_csv(OUT / "all_metrics_long.csv", index=False)
class_df.to_csv(OUT / "classwise_iou.csv", index=False)
status_df.to_csv(OUT / "run_status.csv", index=False)

print("\n\nEVALUATION SUMMARY")
display(status_df)

if not results_df.empty:
    display(results_df)

    # One row per model, separate columns for every split and metric.
    wide_df = results_df.pivot(
        index="Model",
        columns="Split",
        values=["mIoU", "PA", "mPA", "FWIoU"]
    )
    wide_df.columns = [
        f"{metric}_{split}" for metric, split in wide_df.columns
    ]
    wide_df = wide_df.reset_index()
    wide_df.to_csv(OUT / "all_models_comparison_table.csv", index=False)

    print("\nALL-MODEL COMPARISON TABLE")
    display(wide_df)

print("\nOutputs saved in:", OUT)



MODEL: E0 | PROJECT: /home/cse-sdpl/Downloads/CMX_Original
Checkpoint: /home/cse-sdpl/Downloads/CMX_Original/log_TDD_mit_b2/checkpoint/epoch-100.pth

Evaluating E0 on train...
mIoU: 92.72
PA: 99.535
mPA: 96.772
FWIoU: 99.092
Saved log: /home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARISON/train/E0_train.log

Evaluating E0 on val...
mIoU: 62.726
PA: 97.732
mPA: 72.982
FWIoU: 95.784
Saved log: /home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARISON/val/E0_val.log

Evaluating E0 on test...
mIoU: 58.923
PA: 98.227
mPA: 70.841
FWIoU: 96.777
Saved log: /home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARISON/test/E0_test.log

MODEL: E1 | PROJECT: /home/cse-sdpl/Downloads/CMX_E1
Checkpoint: /home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2_e1/checkpoint/epoch-100.pth

Evaluating E1 on train...
mIoU: 93.043
PA: 99.557
mPA: 96.913
FWIoU: 99.134
Saved log: /home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARISON/train/E1_train.log

Evaluating E1 on val...
mIoU: 60.944
PA: 97.684
mPA: 70.88
FWIoU: 95.678
Saved log: /home/

,Model,Split,Status,Project
0,E0,train,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_Original
1,E0,val,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_Original
2,E0,test,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_Original
3,E1,train,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_E1
4,E1,val,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_E1
5,E1,test,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_E1
6,E2,train,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_E2
7,E2,val,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_E2
8,E2,test,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_E2
9,E3,train,Completed; metrics parsed,/home/cse-sdpl/Downloads/CMX_E3


,Model,Split,Checkpoint,mIoU,PA,mPA,FWIoU,Log
0,E0,train,/home/cse-sdpl/Downloads/CMX_Original/log_TDD_...,92.720,99.535,96.772,99.092,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
1,E0,val,/home/cse-sdpl/Downloads/CMX_Original/log_TDD_...,62.726,97.732,72.982,95.784,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
2,E0,test,/home/cse-sdpl/Downloads/CMX_Original/log_TDD_...,58.923,98.227,70.841,96.777,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
3,E1,train,/home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2...,93.043,99.557,96.913,99.134,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
4,E1,val,/home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2...,60.944,97.684,70.880,95.678,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
5,E1,test,/home/cse-sdpl/Downloads/CMX_E1/log_TDD_mit_b2...,56.786,98.176,65.979,96.664,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
6,E2,train,/home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2...,92.646,99.528,96.752,99.080,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
7,E2,val,/home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2...,62.325,97.731,72.627,95.773,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
8,E2,test,/home/cse-sdpl/Downloads/CMX_E2/log_TDD_mit_b2...,57.830,98.224,67.892,96.768,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...
9,E3,train,/home/cse-sdpl/Downloads/CMX_E3/log_TDD_mit_b2...,94.915,99.670,97.607,99.351,/home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARI...



ALL-MODEL COMPARISON TABLE


,Model,mIoU_test,mIoU_train,mIoU_val,PA_test,PA_train,PA_val,mPA_test,mPA_train,mPA_val,FWIoU_test,FWIoU_train,FWIoU_val
0,E0,58.923,92.720,62.726,98.227,99.535,97.732,70.841,96.772,72.982,96.777,99.092,95.784
1,E1,56.786,93.043,60.944,98.176,99.557,97.684,65.979,96.913,70.880,96.664,99.134,95.678
2,E2,57.830,92.646,62.325,98.224,99.528,97.731,67.892,96.752,72.627,96.768,99.080,95.773
3,E3,57.894,94.915,61.961,98.247,99.670,97.723,67.030,97.607,71.182,96.792,99.351,95.739
4,E4,58.086,96.011,62.211,98.258,99.738,97.758,66.617,98.149,71.387,96.816,99.483,95.808
5,E5,58.379,96.611,61.947,98.250,99.776,97.716,67.820,98.424,70.970,96.795,99.557,95.732



Outputs saved in: /home/cse-sdpl/Downloads/CMX_ALL_MODEL_COMPARISON


In [58]:
import pandas as pd
from pathlib import Path

# Output folder
out = Path.home() / "Downloads" / "CMX_ALL_MODEL_COMPARISON"
out.mkdir(parents=True, exist_ok=True)

# Copy the already-created files into Downloads
source_xlsx = Path("/mnt/data/CMX_ALL_MODEL_COMPARISON.xlsx")
source_csv = Path("/mnt/data/CMX_ALL_MODEL_COMPARISON.csv")

if source_xlsx.exists():
    df = pd.read_excel(source_xlsx)
    df.to_excel(out / "CMX_ALL_MODEL_COMPARISON.xlsx", index=False)
    df.to_csv(out / "CMX_ALL_MODEL_COMPARISON.csv", index=False, float_format="%.3f")
    print("Excel:", out / "CMX_ALL_MODEL_COMPARISON.xlsx")
    print("CSV:  ", out / "CMX_ALL_MODEL_COMPARISON.csv")
else:
    print("Source Excel file isn't accessible from your local Jupyter environment.")
    print("Download the Excel file from this chat first.")

Source Excel file isn't accessible from your local Jupyter environment.
Download the Excel file from this chat first.


In [59]:
from pathlib import Path
import shutil

downloads = Path.home() / "Downloads"
source = downloads / "CMX_ALL_MODEL_COMPARISON.xlsx"
destination = downloads / "CMX_ALL_MODEL_COMPARISON" / source.name

destination.parent.mkdir(parents=True, exist_ok=True)

if source.exists():
    shutil.copy2(source, destination)
    print("Saved to:", destination)
else:
    print("First download CMX_ALL_MODEL_COMPARISON.xlsx into your Downloads folder.")

First download CMX_ALL_MODEL_COMPARISON.xlsx into your Downloads folder.


In [62]:
from pathlib import Path
import subprocess
import os

ROOT = Path("/home/cse-sdpl/Downloads/CMX_SCR")
PREDICTIONS = ROOT / "predictions_SCR_test"

assert (ROOT / "eval.py").is_file(), "eval.py not found"
assert (
    ROOT / "log_TDD_mit_b2_scr/checkpoint/epoch-100.pth"
).is_file(), "Final SCR checkpoint not found"

if PREDICTIONS.exists() and any(PREDICTIONS.iterdir()):
    raise RuntimeError(
        f"Prediction directory is not empty: {PREDICTIONS}\n"
        "Choose a fresh output directory to avoid mixing predictions."
    )

os.chdir(ROOT)

command = [
    "conda", "run", "--no-capture-output", "-n", "cmx",
    "python", "eval.py",
    "-e", "100",
    "-d", "0",
    "--split", "test",
    "-p", "predictions_SCR_test",
]

print("Evaluating SCR epoch 100 on the TDD test split...")
print("Command:", " ".join(command))

result = subprocess.run(command, cwd=str(ROOT), check=False)

print("\nEvaluation exit code:", result.returncode)
if result.returncode == 0:
    print("Evaluation process completed. Capture the reported metrics.")
else:
    print("Evaluation failed. Share the error output before retrying.")


RuntimeError: Prediction directory is not empty: /home/cse-sdpl/Downloads/CMX_SCR/predictions_SCR_test
Choose a fresh output directory to avoid mixing predictions.